# HEILO Faísca 2.2: destilação por logits (aprende a "pensar" como o professor)

**A ideia (do código que você trouxe, adaptada para a HEILO):** em vez de aprender só o texto certo, a Faísca aprende a **distribuição de probabilidades** de um professor maior. Ela vê não só "a palavra certa é X", mas também "Y e Z também eram prováveis". Isso ensina mais por exemplo lido.

**Professor:** `Qwen/Qwen2.5-1.5B`, licença Apache-2.0, 3× maior que o Qwen 0,5B e bem melhor em português. Ele **só ensina**: os pesos da HEILO continuam 100% dela, e o professor não fica nela.

**As adaptações para a HEILO:**
1. A HEILO não é um modelo do Hugging Face, então ela é carregada do checkpoint próprio (o da Faísca 2.1 no Drive).
2. **Vocabulários diferentes** (16 mil × 151 mil). As posições são **alinhadas pelo caractere**, e cada token do Qwen é **projetado** no token da HEILO que escreve o mesmo texto.
3. Perda = 50% o texto real + 50% a distribuição do professor, com temperatura 2.

**Antes:** a **Faísca 2.1 precisa ter terminado**, porque ela é a base. GPU T4 e o secret `GITHUB_TOKEN`. Depois, `Executar tudo`. Leva cerca de 2h30.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v2.2"
PROFESSOR = "Qwen/Qwen2.5-1.5B"      # Apache-2.0; só ensina, não entra nos pesos da HEILO
MINUTOS_DESTILACAO = 90
LOTE = 8
import torch, time, os, sys, json, shutil, subprocess, itertools
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Drive + código
from google.colab import drive, userdata
drive.mount("/content/drive")
PASTA_V1 = "/content/drive/MyDrive/heilo_seed_v1"
PASTA = "/content/drive/MyDrive/heilo_seed_v2"
assert os.path.exists(f"{PASTA}/pretreino.pt"), "Não achei o checkpoint da Faísca 2.1 no Drive (heilo_seed_v2/pretreino.pt)."
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run("cd /content/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers transformers accelerate
from pathlib import Path
from heilo.training import seed_v1, pretrain, destilacao
from heilo.models.seed.tokenizer import BPETokenizer
from heilo.models.seed.gpt import MiniGPT, GPTConfig

In [ ]:
# 3) Aluno: a Faísca 2.1 (antes do ajuste de conversa), carregada do Drive
tok = BPETokenizer.from_json(Path(PASTA_V1, "tokenizer.json").read_text(encoding="utf-8"))
base = Path(PASTA) / "pretreino.pt"
copia = Path(PASTA) / "pretreino_antes_destilacao.pt"
if not copia.exists():
    shutil.copy(base, copia)
ck_dest = Path(PASTA) / "destilacao.pt"
st = torch.load(ck_dest if ck_dest.exists() else copia, map_location="cuda", weights_only=False)
cfg = GPTConfig(**st["config"])
aluno = MiniGPT(cfg).cuda()
aluno.load_state_dict(st["modelo"]); aluno.tokenizer = tok
feitos = st.get("minutos_destilacao", 0.0)
print(f"aluno: {aluno.n_params()/1e6:.1f} M parâmetros | {'retomando' if ck_dest.exists() else 'começando'} "
      f"({feitos:.0f} min já destilados)")

In [ ]:
# 4) Professor + mapa de vocabulário (Qwen → HEILO)
from transformers import AutoModelForCausalLM, AutoTokenizer
tok_prof = AutoTokenizer.from_pretrained(PROFESSOR)
professor = AutoModelForCausalLM.from_pretrained(PROFESSOR, torch_dtype=torch.float16).cuda().eval()
for p in professor.parameters():
    p.requires_grad = False
mapa_arq = Path(PASTA) / "mapa_qwen_heilo.pt"
if mapa_arq.exists():
    mapa = torch.load(mapa_arq)
else:
    t0 = time.time()
    mapa = destilacao.mapa_vocab(destilacao.TokHF(tok_prof).textos_dos_tokens(), destilacao.TokHeilo(tok))
    torch.save(mapa, mapa_arq); print(f"mapa pronto em {time.time()-t0:.0f}s")
print(f"tokens do professor com par na HEILO: {(mapa >= 0).float().mean():.1%}")
exemplo = "A capital do Brasil é Brasília, no Distrito Federal."
_, _, pares = destilacao.preparar_exemplo(exemplo, destilacao.TokHeilo(tok), destilacao.TokHF(tok_prof), tok.doc, 512)
print("posições alinhadas no exemplo:", len(pares), "de", len(tok.encode(exemplo)))

## Destilação (pelo relógio; salva no Drive a cada 500 passos)

In [ ]:
# 5) Textos em português (Wikipédia, em pedaços de ~1.500 caracteres)
def pedacos():
    for art in seed_v1.textos_wikipedia(10**12, log=lambda s: None):
        for i in range(0, len(art), 1500):
            if len(art) - i > 300:
                yield art[i:i + 1500]
textos = pedacos()

def salvar(passo):
    torch.save({"modelo": aluno.state_dict(), "config": cfg.__dict__,
                "minutos_destilacao": feitos + (time.time() - t_ini) / 60}, ck_dest)

t_ini = time.time()
restante = max(0.0, MINUTOS_DESTILACAO - feitos)
res_d = destilacao.destilar(aluno, tok, textos, professor, tok_prof, mapa, minutos=restante, lote=LOTE,
                            max_tokens=512, lr=1e-4, lr_min=1e-5, alfa=0.5, beta=0.5, temperatura=2.0,
                            salvar=salvar, salvar_cada=500)
salvar(res_d["passos"])
print("destilação:", res_d["passos"], "passos |", res_d["hist"][-1] if res_d["hist"] else "")
del professor; torch.cuda.empty_cache()

## Ajuste de conversa + avaliação

In [ ]:
# 6) Corpus local (para o ajuste misturado) + SFT com os exemplos aprovados
corpus_dir = Path(PASTA_V1) / "corpus_grande"
local = Path("/content/corpus_local")
if not (local / "meta.json").exists():
    shutil.copytree(corpus_dir, local, dirs_exist_ok=True)
from heilo.training.pipeline import TrainingPipeline
pipe = TrainingPipeline()
split = seed_v1.dividir_sft(pipe)
print(split["manifest"])
res_sft = pretrain.ajustar_conversa(aluno, split["treino"], split["val"], passos=3000, lote=32,
                                    lr=5e-5, paciencia=5, pasta_corpus=local, peso_texto=0.5)
arq = pretrain.exportar(aluno, Path(PASTA) / f"heilo_seed_{NOME_VERSAO}.pt",
                        {"versao": NOME_VERSAO, "base": "v2.1", "destilacao": {"professor": PROFESSOR,
                         "licenca_professor": "Apache-2.0", **res_d}, "sft": res_sft})

In [ ]:
# 7) Prova na mesma régua + decisão
from heilo.training.ciclos import CiclosSeed
from heilo.models.seed.gpt import MiniGPTChat
ciclos = CiclosSeed(pipe)
dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                        {"base": "v2.1", "destilacao": {"professor": PROFESSOR, "passos": res_d["passos"],
                                                        "minutos": res_d["minutos"]},
                         "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val")}},
                        origem=f"Faísca 2.1 + destilação por logits de {PROFESSOR} (alinhamento por caractere) + SFT")
print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
for q in ["oi, tudo bem?", "quem é você?", "qual é a capital de portugal?", "o que é uma função em python?",
          "me fala sobre o Brasil", "o que é energia solar?"]:
    print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7,
                                                  penalidade=1.2, sem_repetir=4), "\n")

In [ ]:
# 8) Salvar no GitHub
!cd /content/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd /content/heilo && git add -f models/seed/versions/{NOME_VERSAO} data/training/versions.json data/training/eval && git add models/seed/weights && git commit -q -m "HEILO Faísca {NOME_VERSAO}: destilação de {PROFESSOR} (Colab)" || echo "nada novo"
!cd /content/heilo && git stash -u -q; git pull --rebase && git push && echo "ENVIADO PARA O GITHUB"